# BNNR Cooking Round 1 — Butterfly classifier that you can actually understand 🦋

**Bulletproof Neural Network Recipe** — automated augmentation search with explainability for PyTorch.

Summer is slowly coming, so instead of another dry benchmark, let's train on butterflies and make it useful for your CV too ☀️🦋

If you are a student, you probably know this scenario: you train a model, get one number (accuracy), and still do not really know what happened. Why does it fail on some classes? Which image regions does it use? Which augmentation actually helps?

This notebook is built exactly for that problem. We train on the real [Kaggle butterfly image classification dataset](https://www.kaggle.com/datasets/phucthaiv02/butterfly-image-classification) and then go from **"I have a model"** to **"I understand this model"**.

What you will get from one run:

- **Training + augmentation experiment** (including ICD and AICD, driven by saliency maps)
- **XAI overlays** (so you can see where the model "looks")
- **`report.json`** (clean summary for CV/portfolio)
- **`events.jsonl`** (full training history)

> Colab note: you cannot open `localhost` from your own browser, so in Colab treat dashboard as optional and focus on the saved artifacts (`report.json`, XAI images, `events.jsonl`).

**Runtime:** around **10–25 minutes** on Colab GPU (T4-like). CPU can be much slower.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bnnr-team/bnnr/blob/main/examples/classification/bnnr_cooking_round1_butterfly.ipynb)


## 1. Installation

Install BNNR with the dashboard extra, plotting, and Kaggle download support.


In [ ]:
%pip install -q "bnnr[dashboard]" matplotlib kagglehub pillow


## 2. Kaggle credentials

`kagglehub` needs Kaggle API credentials to download the butterfly dataset:

- **Local:** place `kaggle.json` under `~/.kaggle/` (see [Kaggle API docs](https://www.kaggle.com/docs/api)).
- **Colab:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Secrets, or upload `kaggle.json`.

This notebook uses only the real butterfly dataset (no synthetic fallback).


## 3. Imports and demo switches


In [ ]:
import glob
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from IPython.display import Image as IPImage
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

from bnnr import (
    BasicAugmentation,
    BNNRConfig,
    BNNRTrainer,
    ChurchNoise,
    SimpleTorchAdapter,
    TorchvisionAugmentation,
    start_dashboard,
)
from bnnr.icd import AICD, ICD
from bnnr.xai_cache import XAICache

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_COLAB = "google.colab" in sys.modules

# Optional caps (0 = use all after split). Example: export BNNR_COOKING_MAX_TRAIN=800
MAX_TRAIN = int(os.environ.get("BNNR_COOKING_MAX_TRAIN", "0") or "0")
MAX_VAL = int(os.environ.get("BNNR_COOKING_MAX_VAL", "0") or "0")

REPORT_DIR = Path("reports/butterfly_cooking")
IMG_SIZE = 128
DASHBOARD_PORT = int(os.environ.get("BNNR_COOKING_DASHBOARD_PORT", "8860"))

print(f"Device: {DEVICE}  IS_COLAB: {IS_COLAB}")


## 4. Dataset — real butterfly data from Kaggle


In [ ]:
import csv


def _has_class_subdirs(folder: Path) -> bool:
    """True if *folder* has subdirectories that look like class folders with images."""
    if not folder.is_dir():
        return False
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    n_ok = 0
    for ch in sorted(folder.iterdir()):
        if not ch.is_dir():
            continue
        imgs = [p for p in ch.iterdir() if p.suffix.lower() in exts]
        if len(imgs) >= 1:
            n_ok += 1
        if n_ok >= 2:
            return True
    return False


def _safe_label(label: str) -> str:
    return "_".join(label.strip().lower().split())


def _materialize_imagefolder_from_csv(root: Path) -> Path:
    """Create ImageFolder-style directories from Kaggle CSV labels (symlinks, no duplication)."""
    source_train = root / "train"
    csv_path = root / "Training_set.csv"
    imagefolder_root = root / "imagefolder_from_csv"

    if imagefolder_root.exists() and _has_class_subdirs(imagefolder_root):
        return imagefolder_root

    if not source_train.is_dir() or not csv_path.is_file():
        raise RuntimeError("Expected Kaggle layout with train/ and Training_set.csv.")

    imagefolder_root.mkdir(parents=True, exist_ok=True)

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    for row in rows:
        filename = row["filename"]
        label = _safe_label(row["label"])
        src = source_train / filename
        if not src.is_file():
            continue
        class_dir = imagefolder_root / label
        class_dir.mkdir(parents=True, exist_ok=True)
        dst = class_dir / filename
        if dst.exists():
            continue
        try:
            dst.symlink_to(src)
        except OSError:
            # Fallback for filesystems that do not support symlinks.
            import shutil

            shutil.copy2(src, dst)

    return imagefolder_root


def _find_imagefolder_train_val_dirs(download_root: Path) -> tuple[Path | None, Path | None]:
    """Return ImageFolder-ready train dir; val may be absent and then we split train."""
    candidates = [download_root]
    try:
        candidates.extend([p for p in download_root.iterdir() if p.is_dir()])
    except FileNotFoundError:
        pass

    for base in candidates:
        t = base / "train"
        if t.is_dir() and _has_class_subdirs(t):
            val_dir = None
            for vn in ("val", "valid", "validation", "test"):
                v = base / vn
                if v.is_dir() and _has_class_subdirs(v):
                    val_dir = v
                    break
            return t, val_dir

    # Kaggle butterfly format: flat train/ plus Training_set.csv
    for base in candidates:
        if (base / "train").is_dir() and (base / "Training_set.csv").is_file():
            prepared = _materialize_imagefolder_from_csv(base)
            return prepared, None

    return None, None


def download_butterfly_kaggle() -> Path:
    import kagglehub

    return Path(kagglehub.dataset_download("phucthaiv02/butterfly-image-classification"))


class IndexedDataset(Dataset):
    """Maps (image, label) -> (image, label, index) for XAICache / ICD / AICD."""

    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, lbl = self.ds[idx]
        return img, lbl, idx


def prepare_butterfly_loaders():
    transform = transforms.Compose(
        [
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
        ]
    )

    root = download_butterfly_kaggle()
    print("Downloaded dataset root:", root)
    train_dir, val_dir = _find_imagefolder_train_val_dirs(root)
    if train_dir is None:
        raise RuntimeError(
            "Could not build ImageFolder-compatible train data from the Kaggle butterfly dataset."
        )

    if val_dir is None:
        full = datasets.ImageFolder(train_dir, transform=transform)
        targets = np.array(full.targets)
        idx = np.arange(len(full))
        tr_i, va_i = train_test_split(
            idx, test_size=0.2, random_state=SEED, stratify=targets
        )
        train_ds: Dataset = Subset(full, tr_i.tolist())
        val_ds = Subset(full, va_i.tolist())
        num_classes = len(full.classes)
        class_names = list(full.classes)
    else:
        train_ds = datasets.ImageFolder(train_dir, transform=transform)
        val_ds = datasets.ImageFolder(val_dir, transform=transform)
        num_classes = len(train_ds.classes)
        class_names = list(train_ds.classes)

    if MAX_TRAIN > 0 and len(train_ds) > MAX_TRAIN:
        train_ds = Subset(train_ds, list(range(MAX_TRAIN)))
    if MAX_VAL > 0 and len(val_ds) > MAX_VAL:
        val_ds = Subset(val_ds, list(range(MAX_VAL)))

    train_loader = DataLoader(
        IndexedDataset(train_ds), batch_size=32, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        IndexedDataset(val_ds), batch_size=32, shuffle=False, num_workers=0
    )
    print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Classes: {num_classes}")
    return train_loader, val_loader, num_classes, class_names


train_loader, val_loader, NUM_CLASSES, CLASS_NAMES = prepare_butterfly_loaders()


## 5. Model

**Do not** use `transforms.Normalize()` in the input pipeline: BNNR augmentations expect roughly **[0, 1]** pixel tensors. Use **BatchNorm** inside the CNN instead (same pattern as the STL-10 demo).


In [ ]:
class DemoNet(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    @property
    def target_layer(self):
        return self.features[8]


model = DemoNet(num_classes=NUM_CLASSES)
model = model.to(DEVICE)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES[:5]}{'...' if len(CLASS_NAMES) > 5 else ''}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 6. Adapter, config, and augmentation candidates


In [ ]:
adapter = SimpleTorchAdapter(
    model=model,
    criterion=nn.CrossEntropyLoss(label_smoothing=0.05),
    optimizer=torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4),
    target_layers=[model.target_layer],
    device=DEVICE,
)

# Student-friendly defaults: still meaningful, but not extremely long.
m_epochs = 4
max_iterations = 3

config = BNNRConfig(
    m_epochs=m_epochs,
    max_iterations=max_iterations,
    selection_metric="accuracy",
    selection_mode="max",
    device=DEVICE,
    xai_enabled=True,
    xai_method="opticam",
    report_dir=str(REPORT_DIR),
    report_preview_size=384,
    report_xai_size=384,
    event_log_enabled=True,
    save_checkpoints=True,
    candidate_pruning_enabled=True,
    candidate_pruning_warmup_epochs=1,
    candidate_pruning_relative_threshold=0.55,
    seed=SEED,
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
xai_cache_dir = REPORT_DIR / "xai_cache"
xai_cache_dir.mkdir(parents=True, exist_ok=True)

print("Precomputing XAI saliency cache for ICD / AICD ...")
xai_cache = XAICache(cache_dir=str(xai_cache_dir))
n_cache = len(train_loader.dataset)
written = xai_cache.precompute_cache(
    model=model,
    train_loader=train_loader,
    target_layers=[model.target_layer],
    n_samples=n_cache,
    method="opticam",
    show_progress=True,
)
print(f"Cached {written} saliency maps (new writes; skips existing unless forced).")

augmentations = [
    ChurchNoise(probability=0.5, intensity=0.45, random_state=SEED),
    BasicAugmentation(probability=0.5, intensity=0.5, random_state=SEED + 1),
    TorchvisionAugmentation(
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
        name_override="tv_color_jitter",
        probability=0.5,
        random_state=SEED + 20,
    ),
    TorchvisionAugmentation(
        transforms.RandomHorizontalFlip(p=1.0),
        name_override="tv_hflip",
        probability=0.5,
        random_state=SEED + 21,
    ),
]

icd = ICD(
    model=model,
    target_layers=[model.target_layer],
    threshold_percentile=70.0,
    explainer="opticam",
    use_cuda=(DEVICE != "cpu"),
    cache=xai_cache,
    tile_size=8,
    fill_strategy="gaussian_blur",
    probability=0.5,
    random_state=SEED + 50,
)
icd.name = "icd"
augmentations.append(icd)

aicd = AICD(
    model=model,
    target_layers=[model.target_layer],
    threshold_percentile=70.0,
    explainer="opticam",
    use_cuda=(DEVICE != "cpu"),
    cache=xai_cache,
    tile_size=8,
    fill_strategy="gaussian_blur",
    probability=0.5,
    random_state=SEED + 51,
)
aicd.name = "aicd"
augmentations.append(aicd)

print(f"{len(augmentations)} augmentation candidates:")
for i, aug in enumerate(augmentations, 1):
    tag = " [XAI-driven]" if aug.name in ("icd", "aicd") else ""
    print(f"  {i:2d}. {aug.name:<22s}  p={aug.probability:.2f}{tag}")


## 7. Training (and dashboard behavior on Colab)

`start_dashboard` can run a local dashboard server. This is useful on your own machine.

In **Colab**, browser access to your local machine `localhost` is not available, so you should treat dashboard as optional there and focus on artifacts produced by the run:
- `report.json`
- `events.jsonl`
- XAI image files under `artifacts/xai/`

`BNNRTrainer.run()` still performs the full baseline + augmentation-branch process and writes all these outputs.


In [ ]:
# Start dashboard only outside Colab (Colab cannot expose your local localhost in a normal way).
if not IS_COLAB:
    dashboard_url = start_dashboard(
        REPORT_DIR.resolve(),
        port=DASHBOARD_PORT,
        auto_open=False,
        build_frontend=False,
    )
    print("Dashboard (live during training):", dashboard_url)
else:
    print("Running in Colab: skipping local dashboard server view.")

trainer = BNNRTrainer(adapter, train_loader, val_loader, augmentations, config)
result = trainer.run()
print("Training finished. Report:", result.report_json_path)


## 8. Results — metrics and selected augmentations


In [ ]:
print("=" * 60)
print("  BNNR — butterfly run summary")
print("=" * 60)
print(f"  Best augmentation path : {result.best_path}")
print(f"  Best metrics           : {result.best_metrics}")
print(f"  Selected augmentations : {result.selected_augmentations}")
print(f"  Report JSON            : {result.report_json_path}")
print("=" * 60)

if result.selected_augmentations:
    print("\nStacked augmentations (best-first along the path):")
    for i, name in enumerate(result.selected_augmentations, 1):
        tag = "  <- XAI-driven" if name in ("icd", "aicd") else ""
        print(f"  {i}. {name}{tag}")


## 9. Report JSON highlights


In [ ]:
run_dir = result.report_json_path.parent
with open(result.report_json_path, encoding="utf-8") as f:
    rep = json.load(f)

# Top-level keys vary by version; print a compact slice.
print("Top-level keys:", sorted(rep.keys())[:20])
for key in ("best_metrics", "selection_metric", "selected_augmentations", "best_path"):
    if key in rep:
        print(f"{key}: {rep[key]}")

# Optional: CLI summary (same as `python -m bnnr report ...`)
try:
    out = subprocess.run(
        [sys.executable, "-m", "bnnr", "report", str(result.report_json_path), "--format", "summary"],
        capture_output=True,
        text=True,
        check=True,
        timeout=120,
    )
    print("\n--- bnnr report --format summary ---\n")
    print(out.stdout)
except Exception as exc:  # noqa: BLE001 — notebook UX
    print("(CLI report skipped)", exc)


## 10. XAI overlays and augmentation previews


In [ ]:
xai_files = sorted(glob.glob(str(run_dir / "artifacts" / "xai" / "**" / "*.png"), recursive=True))
if xai_files:
    print(f"Found {len(xai_files)} XAI PNGs. Showing up to 6:")
    for f in xai_files[:6]:
        print(Path(f).name)
        display(IPImage(filename=f, width=560))
else:
    print("No XAI PNGs found under artifacts/xai.")

preview_files = sorted(glob.glob(str(run_dir / "artifacts" / "candidate_previews" / "*.png")))
if not preview_files:
    preview_files = sorted(glob.glob(str(run_dir / "artifacts" / "augmentation_previews" / "*.png")))
if preview_files:
    print(f"\nPreview images ({len(preview_files)}), showing up to 3:")
    for f in preview_files[:3]:
        display(IPImage(filename=f, width=560))


## 11. Events log (`events.jsonl`)


In [ ]:
from bnnr.events import load_events

events_path = run_dir / "events.jsonl"
if events_path.exists():
    events = load_events(events_path)
    counts: dict[str, int] = {}
    for e in events:
        t = str(e.get("type", "unknown"))
        counts[t] = counts.get(t, 0) + 1
    print(f"Total events: {len(events)}")
    for t, c in sorted(counts.items()):
        print(f"  {t:<40} {c}")
else:
    print("No events.jsonl at", events_path)


## 12. Static dashboard export (for a website)

To ship a **non-interactive** snapshot of the dashboard UI with your article or portfolio page, run **python3 -m bnnr dashboard export --run-dir RUN_DIR --out exported_dashboard** in a terminal.

Use the timestamped directory that contains your **report.json** as **RUN_DIR** (the `run_dir` variable in this notebook is that path).


## Next steps

- Deeper augmentation catalogue: [Augmentations guide notebook](../bnnr_augmentations_guide.ipynb)
- Full STL-10 walkthrough: [Classification demo](./bnnr_classification_demo.ipynb)
- Bring your own folders: [Custom data notebook](../bnnr_custom_data.ipynb)
